# Restaurant Analytics Agent - Data Foundation

This notebook creates Unity Catalog tables that power a ReAct-pattern agent with four tools:
- **Business Lookup**: Retrieve restaurant ratings, review counts, and metadata
- **Vector Search**: Semantic search over customer reviews (requires review text embeddings)
- **Competitor Comparison**: Compare ratings and trends across restaurants
- **Complaint/Opportunity Extraction**: Identify recurring themes in reviews

**Data pipeline**: Ingest compressed Google Local data → filter San Diego Italian restaurants → create Delta tables in Unity Catalog

**Agent stack**: 
- LLM: Meta Llama 3.3 70B Instruct (Databricks Foundation Model API)
- Embeddings: GTE Large (En) for vector search
- Framework: LangGraph for tool routing

In [0]:
# Install required packages
%pip install databricks-vectorsearch
%pip install databricks-vectorsearch openai
dbutils.library.restartPython()

---
## 1. Setup Unity Catalog Structure

Create the catalog, schema, and volume for staging source data and storing agent tables.

**UC namespace**: `main.aai510_final_agent`  
**Volume**: Stores compressed source files for reusable Spark reads (no local `wget` downloads)  
**Source data**:
- `meta-California.json.gz` - Business metadata (name, address, categories, location)
- `rating-California.csv.gz` - User ratings (gmap_id, user_id, rating, timestamp)

In [0]:
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "aai510_final_agent"
VOLUME = "aai510_final_raw"

# Source URLs
META_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/meta-California.json.gz"
RATING_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/rating-California.csv.gz"

# Create Unity Catalog objects
print("=" * 60)
print("Creating Unity Catalog structure...")
print("=" * 60)

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Volume paths for staged files
META_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/meta-California.json.gz"
RATING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/rating-California.csv.gz"

# Helper function to check whether files already exist in Volume
def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print(f"\nStaging compressed files to Volume: {CATALOG}.{SCHEMA}.{VOLUME}")
print("Checking for existing staged files...\n")

# Stage files into Volume only if needed
if file_exists(META_PATH):
    print(f"✓ Already staged: {META_PATH}")
else:
    dbutils.fs.cp(META_URL, META_PATH)
    print(f"✓ Staged: {META_PATH}")

if file_exists(RATING_PATH):
    print(f"✓ Already staged: {RATING_PATH}")
else:
    dbutils.fs.cp(RATING_URL, RATING_PATH)
    print(f"✓ Staged: {RATING_PATH}")

print(f"\n✓ Unity Catalog setup complete")
print(f"\tCatalog: {CATALOG}")
print(f"\tSchema: {CATALOG}.{SCHEMA}")
print(f"\tVolume: {CATALOG}.{SCHEMA}.{VOLUME}")

---
### 1.1. Inspect Staged Files

Verify that files were successfully staged and examine their schemas to understand the data structure.

**What we're checking**:
- Total record counts for California dataset
- Schema structure (column names and types)
- Sample records to confirm data quality
- Whether review text is available (needed for Vector Search)

This step helps identify any data quality issues early and confirms which agent tools can be implemented with the current dataset.

In [0]:
# Verify staged files and inspect schema
print("=" * 60)
print("Inspecting Staged Files...")
print("=" * 60)

# Read metadata sample
print("\n1. Business Metadata (meta-California.json.gz)")
print("-" * 60)
meta_df = spark.read.json(META_PATH)
print(f"Total businesses in California: {meta_df.count():,}")
print("\nSample records:")
display(meta_df.limit(3))

# Read ratings sample
print("\n2. Ratings Data (rating-California.csv.gz)")
print("-" * 60)
ratings_df = spark.read.option("header", "true").csv(RATING_PATH)
print(f"Total ratings in California: {ratings_df.count():,}")
print("\nSample records:")
display(ratings_df.limit(3))

---
## 2. Filter San Diego Italian Restaurants

Read business metadata and filter for San Diego Italian restaurants using Spark SQL expressions. This creates the foundational `restaurants` table that powers the Business Lookup tool and defines which reviews are relevant for analysis.

**Filters applied**:
- **Geography**: Address contains "San Diego, CA"
- **Category**: Business category includes "Restaurant" AND "Italian"

**Output table**: `main.aai510_final_agent.restaurants`

In [0]:
# Read and filter business metadata
print("=" * 60)
print("Filtering San Diego Italian restaurants...")
print("=" * 60)

# Read metadata with schema inference
meta_df = spark.read.json(META_PATH)

# Filter logic (mimics original regex + category matching)
restaurants_df = (
    meta_df
    # Convert category array to searchable string
    .withColumn("category_text", 
                F.lower(F.concat_ws(" ", F.coalesce(F.col("category"), F.array()))))
    # Normalize address for matching
    .withColumn("address_lower", 
                F.lower(F.coalesce(F.col("address"), F.lit(""))))
    # Apply filters
    .filter(F.col("address_lower").rlike(r",\s*san diego,\s*ca\b"))
    .filter(F.col("category_text").contains("restaurant"))
    .filter(F.col("category_text").contains("italian"))
    # Select clean schema for agent tools
    .select(
        "gmap_id",
        F.col("name").alias("business_name"),
        "address",
        "category",
        "latitude",
        "longitude"
    )
    .dropna(subset=["gmap_id"])
    .dropDuplicates(["gmap_id"])
)

# Preview results
restaurant_count = restaurants_df.count()
print(f"\n✓ Found {restaurant_count:,} San Diego Italian restaurants")

print("\nSample restaurants:")
display(restaurants_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurants"

print("=" * 60)
print(f"\nWriting to Unity Catalog: {table_name}")
print("=" * 60)

restaurants_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

---
## 3. Join Reviews to Filtered Restaurants

Read the ratings CSV and join to our filtered San Diego Italian restaurants. This creates the `reviews` table that contains all customer ratings for our target restaurants.

**Output table**: `main.aai510_final_agent.reviews`  
**Schema**: `gmap_id`, `user_id`, `rating`, `timestamp`

**Note**: This dataset contains ratings only (no review text). For vector search tools, review text will need to be sourced separately.

In [0]:
# Read and join reviews to filtered restaurants
print("=" * 60)
print("Loading and filtering reviews...")
print("=" * 60)

# Read ratings CSV with header
ratings_df = (
    spark.read
    .option("header", "true")
    .csv(RATING_PATH)
    .withColumnRenamed("business", "gmap_id")
    .withColumnRenamed("user", "user_id")
)

print(f"Total ratings in California: {ratings_df.count():,}")

# Load restaurant filter list
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
restaurant_ids = restaurants_df.select("gmap_id")

print(f"Filtering to {restaurant_ids.count():,} San Diego Italian restaurants...")

# Inner join to get only reviews for our target restaurants
reviews_df = (
    ratings_df
    .join(restaurant_ids, on="gmap_id", how="inner")
    .select("gmap_id", "user_id", "rating", "timestamp")
)

review_count = reviews_df.count()
print(f"\n✓ Found {review_count:,} reviews for San Diego Italian restaurants")

print("\nSample reviews:")
display(reviews_df.limit(5))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.reviews"
print(f"\nWriting to Unity Catalog: {table_name}")
reviews_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

# Summary stats
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Restaurants: {restaurant_ids.count():,}")
print(f"Reviews: {review_count:,}")
print(f"Avg reviews per restaurant: {review_count / restaurant_ids.count():.1f}")

---
## 4. Create Restaurant Metrics Table

Aggregate review data to create metrics for each restaurant. This table powers the **Business Lookup** and **Competitor Comparison** agent tools.

**Output table**: `main.aai510_final_agent.restaurant_metrics`  
**Metrics computed**:
- Average rating
- Total review count
- Rating distribution (count by 1-5 stars)
- Latest review timestamp
- Restaurant metadata (name, address, location)

In [0]:
# Create aggregated metrics from reviews
print("=" * 60)
print("Computing restaurant metrics...")
print("=" * 60)

# Load reviews and restaurants
reviews_df = spark.table(f"{CATALOG}.{SCHEMA}.reviews")
restaurants_df = spark.table(f"{CATALOG}.{SCHEMA}.restaurants")

# Aggregate metrics per restaurant
metrics_df = (
    reviews_df
    .groupBy("gmap_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.max("timestamp").alias("latest_review_timestamp"),
        # Rating distribution: count by rating value (1-5 stars)
        F.sum(F.when(F.col("rating") == "1", 1).otherwise(0)).alias("rating_1_count"),
        F.sum(F.when(F.col("rating") == "2", 1).otherwise(0)).alias("rating_2_count"),
        F.sum(F.when(F.col("rating") == "3", 1).otherwise(0)).alias("rating_3_count"),
        F.sum(F.when(F.col("rating") == "4", 1).otherwise(0)).alias("rating_4_count"),
        F.sum(F.when(F.col("rating") == "5", 1).otherwise(0)).alias("rating_5_count"),
    )
)

# Join with restaurant metadata
restaurant_metrics_df = (
    metrics_df
    .join(restaurants_df, on="gmap_id", how="inner")
    .select(
        "gmap_id",
        "business_name",
        "address",
        "category",
        "latitude",
        "longitude",
        "avg_rating",
        "review_count",
        "latest_review_timestamp",
        "rating_1_count",
        "rating_2_count",
        "rating_3_count",
        "rating_4_count",
        "rating_5_count",
    )
    .orderBy(F.desc("review_count"))
)

print(f"\n✓ Computed metrics for {restaurant_metrics_df.count():,} restaurants")

print("\nTop 10 restaurants by review count:")
display(restaurant_metrics_df.limit(10))

# Write to Unity Catalog
table_name = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
print(f"\nWriting to Unity Catalog: {table_name}")
restaurant_metrics_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"✓ Table created: {table_name}")

print("\n" + "=" * 60)
print("✓ Restaurant metrics table ready for agent tools")
print("=" * 60)

---
## 5. Prepare for Vector Search (Review Text)

The current `reviews` table only contains ratings (1-5 stars) without review text. For **Vector Search** and **Complaint/Opportunity Extraction** tools, we need the actual review text to create embeddings.

**Requirements for Vector Search**:
- Review text field for GTE Large embeddings
- Join reviews to filtered San Diego Italian restaurants
- Create Databricks Vector Search index

**Data source**: `review-California.json.gz` contains full review data including text field

In [0]:
# Prepare Review Text for Theme Retrieval

from pyspark.sql import functions as F

# Review text configuration
REVIEW_TEXT_URL = "https://mcauleylab.ucsd.edu:8443/public_datasets/gdrive/googlelocal/review-California.json.gz"
REVIEW_TEXT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/review-California.json.gz"
REVIEW_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"

print("=" * 60)
print("Preparing review text for Theme Retrieval...")
print("=" * 60)

print(f"\nReview text source: {REVIEW_TEXT_URL}")
print(f"Review text path: {REVIEW_TEXT_PATH}")
print(f"Review text table: {REVIEW_TEXT_TABLE}")

In [0]:
# Check whether review text table already exists
try:
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Found existing table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))
    
    REVIEW_TEXT_READY = True

except Exception:
    print(f"Review text table not found: {REVIEW_TEXT_TABLE}")
    print("Checking whether raw review text file is already staged...")
    
    REVIEW_TEXT_READY = False

In [0]:
# Stage raw review text file only if needed
if not REVIEW_TEXT_READY:
    
    try:
        dbutils.fs.ls(REVIEW_TEXT_PATH)
        print(f"✓ Already staged: {REVIEW_TEXT_PATH}")
        
    except Exception:
        print("Raw review text file not found.")
        print("Staging raw review text file. This may take several minutes...")
        
        dbutils.fs.cp(REVIEW_TEXT_URL, REVIEW_TEXT_PATH)
        
        print(f"✓ Staged: {REVIEW_TEXT_PATH}")

else:
    print("Skipping raw file staging because review text table already exists.")

In [0]:
# Create review text table only if needed
if not REVIEW_TEXT_READY:
    
    print("=" * 60)
    print("Creating reviews_with_text table...")
    print("=" * 60)
    
    # Read raw review text file
    reviews_with_text_raw_df = spark.read.json(REVIEW_TEXT_PATH)
    
    # Get filtered restaurant IDs from Section 2
    restaurant_ids_df = (
        spark.table(f"{CATALOG}.{SCHEMA}.restaurants")
        .select("gmap_id")
        .distinct()
    )
    
    # Keep only written reviews for selected restaurants
    reviews_with_text_filtered_df = (
        reviews_with_text_raw_df
        .join(restaurant_ids_df, on="gmap_id", how="inner")
        .select(
            "gmap_id",
            "user_id",
            "rating",
            "time",
            "text",
            "name"
        )
        .filter(F.col("text").isNotNull())
        .filter(F.length(F.col("text")) > 10)
    )
    
    # Save filtered review text table
    (
        reviews_with_text_filtered_df
        .write
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(REVIEW_TEXT_TABLE)
    )
    
    reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)
    
    print(f"✓ Created table: {REVIEW_TEXT_TABLE}")
    print(f"✓ Total review-text rows: {reviews_with_text_df.count():,}")
    
    display(reviews_with_text_df.limit(5))

else:
    print("Skipping table creation because review text table already exists.")

In [0]:
# Verify review text table
print("=" * 60)
print("Verifying review text table...")
print("=" * 60)

reviews_with_text_df = spark.table(REVIEW_TEXT_TABLE)

print(f"Table: {REVIEW_TEXT_TABLE}")
print(f"Total rows: {reviews_with_text_df.count():,}")

display(
    reviews_with_text_df
    .select("gmap_id", "rating", "text")
    .limit(10)
)

---
##6. Create Agent Tools

Convert the restaurant data pipeline into reusable tools for the agent. These tools allow the agent to answer both structured questions about ratings and review counts and qualitative questions about customer experiences.

1. **Business Lookup**: Retrieve rating, review count, rating distribution, and restaurant metadata.
2. **Competitor Comparison**: Compare restaurants by rating, review volume, and rating distribution.
3. **Theme Retrieval**: Use vector search over review text to retrieve reviews related to themes like service, food quality, ambience, price, wait time, or complaints.

### 6.1. Configure Source Tables

In [0]:
from pyspark.sql import functions as F

# Source tables
RESTAURANT_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.restaurant_metrics"
RESTAURANTS_TABLE = f"{CATALOG}.{SCHEMA}.restaurants"
REVIEWS_WITH_TEXT_TABLE = f"{CATALOG}.{SCHEMA}.reviews_with_text"
REVIEWS_FOR_VECTOR_TABLE = f"{CATALOG}.{SCHEMA}.reviews_for_vector_search"

print("=" * 60)
print("Configuring agent tool source tables...")
print("=" * 60)

print(f"Restaurant metrics table: {RESTAURANT_METRICS_TABLE}")
print(f"Restaurants table: {RESTAURANTS_TABLE}")
print(f"Review text table: {REVIEWS_WITH_TEXT_TABLE}")
print(f"Vector search source table: {REVIEWS_FOR_VECTOR_TABLE}")

### 6.2. Tool 1 - Business Lookup

Objective: Retrieve metrics for a specific restaurant

In [0]:
def business_lookup(restaurant_name: str, limit: int = 5):
    """
    Retrieve rating, review count, rating distribution, and restaurant metadata
    for a specific restaurant.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    results_df = (
        restaurant_metrics_df
        .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
        .select(
            "gmap_id",
            "business_name",
            "address",
            "avg_rating",
            "review_count",
            "rating_1_count",
            "rating_2_count",
            "rating_3_count",
            "rating_4_count",
            "rating_5_count"
        )
        .orderBy(F.desc("review_count"))
        .limit(limit)
    )
    
    results = results_df.toPandas().to_dict(orient="records")
    
    if len(results) == 0:
        return {
            "status": "not_found",
            "message": f"No restaurant found matching '{restaurant_name}'."
        }
    
    return {
        "status": "success",
        "results": results
    }

### 6.3. Tool 2 - Competitor Comparison

Objective: Compare restaurant performance to another restaurant

In [0]:
def competitor_comparison(restaurant_1: str, restaurant_2: str, limit_per_name: int = 3):
    """
    Compare two specific restaurants by rating, review volume, and rating distribution.
    The user provides both restaurant names.
    """
    
    restaurant_metrics_df = spark.table(RESTAURANT_METRICS_TABLE)
    
    restaurant_names = [restaurant_1, restaurant_2]
    all_results = []
    
    for restaurant_name in restaurant_names:
        matches_df = (
            restaurant_metrics_df
            .filter(F.lower(F.col("business_name")).contains(restaurant_name.lower()))
            .select(
                "gmap_id",
                "business_name",
                "address",
                "avg_rating",
                "review_count",
                "rating_1_count",
                "rating_2_count",
                "rating_3_count",
                "rating_4_count",
                "rating_5_count"
            )
            .orderBy(F.desc("review_count"))
            .limit(limit_per_name)
        )
        
        matches = matches_df.toPandas().to_dict(orient="records")
        
        for match in matches:
            match["search_name"] = restaurant_name
        
        all_results.extend(matches)
    
    if len(all_results) == 0:
        return {
            "status": "not_found",
            "message": f"No matching restaurants found for '{restaurant_1}' or '{restaurant_2}'."
        }
    
    return {
        "status": "success",
        "restaurant_1": restaurant_1,
        "restaurant_2": restaurant_2,
        "results": all_results
    }

### 6.4. Prepare Review Text Table for Vector Search

Vector Search needs a stable primary key. This creates a new table with a `review_id`

To keep vector search efficient while making the agent more generalizable, this section indexes all available review text for the two selected comparison restaurants and a random sample of 20 additional San Diego Italian restaurants. This allows the agent to answer the primary comparison question while still supporting theme retrieval for other restaurants in the dataset.

In [0]:
# Create a vector-search-ready review text table with a stable primary key
# Include the two selected restaurants plus a random sample of other restaurants

from pyspark.sql.window import Window

SELECTED_RESTAURANTS = ["cesarina", "parma"]
NUM_RANDOM_RESTAURANTS = 20
MAX_REVIEWS_PER_RANDOM_RESTAURANT = 50

reviews_with_text_df = spark.table(REVIEWS_WITH_TEXT_TABLE)
restaurants_df = spark.table(RESTAURANTS_TABLE)

reviews_joined_df = (
    reviews_with_text_df
    .join(
        restaurants_df.select("gmap_id", "business_name", "address"),
        on="gmap_id",
        how="inner"
    )
    .filter(F.col("text").isNotNull())
    .filter(F.length(F.col("text")) > 10)
)

# Keep all reviews for selected restaurants
selected_reviews_df = (
    reviews_joined_df
    .filter(
        (F.lower(F.col("business_name")).contains("cesarina")) |
        (F.lower(F.col("business_name")).contains("parma"))
    )
)

# Randomly choose additional restaurants, excluding selected restaurants
random_restaurant_ids_df = (
    reviews_joined_df
    .filter(
        ~(
            (F.lower(F.col("business_name")).contains("cesarina")) |
            (F.lower(F.col("business_name")).contains("parma"))
        )
    )
    .select("gmap_id", "business_name")
    .distinct()
    .orderBy(F.rand(seed=42))
    .limit(NUM_RANDOM_RESTAURANTS)
)

# For those random restaurants, take up to N reviews per restaurant
random_reviews_base_df = (
    reviews_joined_df
    .join(
        random_restaurant_ids_df.select("gmap_id"),
        on="gmap_id",
        how="inner"
    )
)

window_by_restaurant = Window.partitionBy("gmap_id").orderBy(F.rand(seed=42))

random_reviews_df = (
    random_reviews_base_df
    .withColumn("random_review_rank", F.row_number().over(window_by_restaurant))
    .filter(F.col("random_review_rank") <= MAX_REVIEWS_PER_RANDOM_RESTAURANT)
    .drop("random_review_rank")
)

# Combine selected restaurants and random sample
reviews_for_vector_df = (
    selected_reviews_df
    .unionByName(random_reviews_df)
    .select(
        F.sha2(
            F.concat_ws(
                "||",
                F.col("gmap_id"),
                F.coalesce(F.col("user_id"), F.lit("unknown_user")),
                F.coalesce(F.col("time").cast("string"), F.lit("unknown_time")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        ).alias("review_id"),
        "gmap_id",
        "business_name",
        "address",
        "rating",
        "time",
        "text"
    )
    .dropDuplicates(["review_id"])
)

(
    reviews_for_vector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(REVIEWS_FOR_VECTOR_TABLE)
)

# Change Data Feed is required for Delta Sync Vector Search indexes
spark.sql(f"""
ALTER TABLE {REVIEWS_FOR_VECTOR_TABLE}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print(f"Created vector-search source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Total rows: {spark.table(REVIEWS_FOR_VECTOR_TABLE).count():,}")

display(
    spark.table(REVIEWS_FOR_VECTOR_TABLE)
    .groupBy("business_name")
    .agg(F.count("*").alias("review_text_count"))
    .orderBy(F.desc("review_text_count"))
)

### 6.5 Initialize Vector Search Client

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

print("Vector Search client initialized.")

### 6.6. Configure Vector Search Endpoint and Index

In [0]:
# Vector Search configuration
VECTOR_SEARCH_ENDPOINT_NAME = "restaurant_reviews_vs_endpoint"
VECTOR_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.restaurant_review_text_index"

# Databricks embedding model endpoint
# If this endpoint is not available in your workspace, replace it with the embedding endpoint your course/workspace provides.
EMBEDDING_MODEL_ENDPOINT_NAME = "databricks-gte-large-en"

print("=" * 60)
print("Configuring Vector Search index...")
print("=" * 60)

print(f"Endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
print(f"Index: {VECTOR_SEARCH_INDEX_NAME}")
print(f"Source table: {REVIEWS_FOR_VECTOR_TABLE}")
print(f"Embedding endpoint: {EMBEDDING_MODEL_ENDPOINT_NAME}")

### 6.7. Create or Connect to Vector Search Endpoint

Create an endpoint/index if needed, otherwise connect to the existing one

In [0]:
# Create Vector Search endpoint if needed

try:
    vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Found existing endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

except Exception:
    print(f"Creating endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    vsc.wait_for_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✓ Created endpoint: {VECTOR_SEARCH_ENDPOINT_NAME}")

### 6.8. Create or Connect to Vector Search Index

In [0]:
# Delete offline / partial Vector Search index

try:
    print(f"Deleting index: {VECTOR_SEARCH_INDEX_NAME}")
    
    vsc.delete_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    print(f"Deleted index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception as e:
    print("Index delete failed or index was already gone.")
    print(e)

In [0]:
# Create Delta Sync index if needed

try:
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    print(f"✓ Found existing index: {VECTOR_SEARCH_INDEX_NAME}")

except Exception:
    print(f"Creating index: {VECTOR_SEARCH_INDEX_NAME}")
    
    review_index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME,
        source_table_name=REVIEWS_FOR_VECTOR_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="review_id",
        embedding_source_column="text",
        embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT_NAME,
        columns_to_sync=[
            "review_id",
            "gmap_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ]
    )
    
    review_index.wait_until_ready()
    print(f"✓ Created index: {VECTOR_SEARCH_INDEX_NAME}")

### 6.9. Sync Vector Search Index

In [0]:
# Sync the index after creating or updating the source table

review_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_SEARCH_INDEX_NAME
)

review_index.sync()
review_index.wait_until_ready()

print("✓ Vector Search index is synced and ready.")

### 6.10. Helper Function to Parse Vector Search Results

In [0]:
def parse_vector_search_results(results):
    """
    Convert Databricks Vector Search response into a list of dictionaries.
    """
    
    columns = [col["name"] for col in results["manifest"]["columns"]]
    rows = results["result"]["data_array"]
    
    parsed_results = []
    
    for row in rows:
        parsed_results.append(dict(zip(columns, row)))
    
    return parsed_results

### 6.11. Tool 3 - Theme Retrieval

In [0]:
def theme_retrieval(restaurant_name: str, theme_query: str, num_results: int = 10):
    """
    Retrieve reviews related to a user-provided theme using Databricks Vector Search.
    This uses semantic similarity search over review text and does not rely on
    hard-coded keywords.
    """
    
    review_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=VECTOR_SEARCH_INDEX_NAME
    )
    
    # Pull more than needed because restaurant filtering happens after retrieval.
    raw_num_results = max(num_results * 5, 25)
    
    results = review_index.similarity_search(
        query_text=theme_query,
        columns=[
            "review_id",
            "business_name",
            "address",
            "rating",
            "time",
            "text"
        ],
        num_results=raw_num_results
    )
    
    parsed_results = parse_vector_search_results(results)
    
    filtered_results = [
        row for row in parsed_results
        if restaurant_name.lower() in row.get("business_name", "").lower()
    ]
    
    filtered_results = filtered_results[:num_results]
    
    if len(filtered_results) == 0:
        return {
            "status": "not_found",
            "message": (
                f"No vector search results found for restaurant '{restaurant_name}' "
                f"and theme query '{theme_query}'."
            ),
            "restaurant_name": restaurant_name,
            "theme_query": theme_query,
            "results": []
        }
    
    return {
        "status": "success",
        "restaurant_name": restaurant_name,
        "theme_query": theme_query,
        "results": filtered_results
    }

### 6.12 Validate All Three Tools

In [0]:
print("=" * 60)
print("Validating agent tools...")
print("=" * 60)

print("\nTool 1: Business Lookup")
business_lookup_result = business_lookup("Cesarina")

if business_lookup_result["status"] == "success":
    display(business_lookup_result["results"])
else:
    print(business_lookup_result["message"])


print("\nTool 2: Competitor Comparison")
comparison_result = competitor_comparison("Cesarina", "Parma Cucina Italiana")

if comparison_result["status"] == "success":
    display(comparison_result["results"])
else:
    print(comparison_result["message"])


print("\nTool 3: Theme Retrieval")
try:
    theme_result = theme_retrieval(
        "Cesarina",
        "reviews about pasta quality and authentic Italian food",
        num_results=5
    )
    
    if theme_result["status"] == "success":
        display(theme_result["results"])
    else:
        print(theme_result["message"])

except Exception as e:
    print("Theme Retrieval failed.")
    print(e)

---
## 7. Create Restaurant Comparison Agent

This section connects the reusable restaurant tools to a tool-calling agent. The agent uses an LLM to interpret the user’s question, choose the appropriate tool, call the tool, and synthesize a final answer using the tool results.

### 7.1. Configure LLM Endpoint

Baseline: Llama 3.3 70B
Comparirson: GPT OSS 20b (cheaper model)

In [0]:
# LLM endpoints used for the restaurant comparison agent

LLAMA_3_3_70B_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
GPT_OSS_20B_ENDPOINT = "databricks-gpt-oss-20b"

AVAILABLE_LLMS = [
    LLAMA_3_3_70B_ENDPOINT,
    GPT_OSS_20B_ENDPOINT
]

# Choose the default LLM for the first agent run
CHOSEN_LLM_ENDPOINT = LLAMA_3_3_70B_ENDPOINT

print("=" * 60)
print("Configuring LLM endpoints...")
print("=" * 60)

print("Available LLM endpoints:")
for endpoint in AVAILABLE_LLMS:
    print(f"- {endpoint}")

print(f"\nDefault LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

### 7.2. Define Tool Metadata

In [0]:
import json
from dataclasses import dataclass
from typing import Callable

@dataclass
class ToolInfo:
    name: str
    spec: dict
    exec_fn: Callable

### 7.3. Define Tool Specifications

In [0]:
# Tool specifications in OpenAI-compatible function-calling format

business_lookup_spec = {
    "type": "function",
    "function": {
        "name": "business_lookup",
        "description": (
            "Retrieve rating, review count, rating distribution, and restaurant metadata "
            "for a specific restaurant."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant to look up."
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of matching restaurants to return."
                }
            },
            "required": ["restaurant_name"]
        }
    }
}


competitor_comparison_spec = {
    "type": "function",
    "function": {
        "name": "competitor_comparison",
        "description": (
            "Compare two specific restaurants by rating, review volume, and rating distribution."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_1": {
                    "type": "string",
                    "description": "Name or partial name of the first restaurant."
                },
                "restaurant_2": {
                    "type": "string",
                    "description": "Name or partial name of the second restaurant."
                },
                "limit_per_name": {
                    "type": "integer",
                    "description": "Maximum number of matches to return per restaurant name."
                }
            },
            "required": ["restaurant_1", "restaurant_2"]
        }
    }
}


theme_retrieval_spec = {
    "type": "function",
    "function": {
        "name": "theme_retrieval",
        "description": (
            "Retrieve reviews related to a user-provided theme using semantic Vector Search "
            "over restaurant review text. Use this for qualitative questions about customer experiences, "
            "including service, food quality, ambiance, price, wait time, complaints, and special occasions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant."
                },
                "theme_query": {
                    "type": "string",
                    "description": (
                        "Natural-language theme to search for in review text. "
                        "For example: 'service and staff friendliness', "
                        "'pasta quality and authentic Italian food', or "
                        "'ambiance for date night'."
                    )
                },
                "num_results": {
                    "type": "integer",
                    "description": "Number of relevant review excerpts to retrieve."
                }
            },
            "required": ["restaurant_name", "theme_query"]
        }
    }
}

### 7.4. Register Tools

In [0]:
TOOL_INFOS = [
    ToolInfo(
        name="business_lookup",
        spec=business_lookup_spec,
        exec_fn=business_lookup
    ),
    ToolInfo(
        name="competitor_comparison",
        spec=competitor_comparison_spec,
        exec_fn=competitor_comparison
    ),
    ToolInfo(
        name="theme_retrieval",
        spec=theme_retrieval_spec,
        exec_fn=theme_retrieval
    )
]

print("=" * 60)
print("Registering agent tools...")
print("=" * 60)

print(f"Registered {len(TOOL_INFOS)} tools:")
for tool in TOOL_INFOS:
    print(f"- {tool.name}")

### 7.5. Create System Prompt

In [0]:
SYSTEM_PROMPT = """
You are a restaurant comparison assistant focused on San Diego Italian restaurants.

Your scope:
- Answer questions about San Diego Italian restaurants using the available restaurant metrics and review-text tools.
- Help users compare restaurants, understand ratings, review counts, rating distributions, and customer review themes.
- Provide recommendations only when they can be grounded in the available tool outputs.

Out-of-scope requests:
- Do not answer questions unrelated to San Diego Italian restaurants.
- Do not provide general homework help, coding help, medical advice, legal advice, financial advice, travel planning, politics, sports, entertainment, or unrelated factual information.
- Do not invent restaurant data, restaurant names, metrics, ratings, review counts, review text, or business recommendations.

If the user asks an out-of-scope question, politely reject it and briefly explain that you can only help with San Diego Italian restaurant comparison questions using the available data.

You have access to three tools:
1. business_lookup: retrieves restaurant-level metrics such as average rating, review count, and rating distribution.
2. competitor_comparison: compares two restaurants using rating, review volume, and rating distribution.
3. theme_retrieval: retrieves semantically relevant review excerpts for qualitative customer-experience themes.

Tool-use rules:
- For structured questions about one restaurant, use business_lookup.
- For structured comparison questions about two restaurants, use competitor_comparison.
- For qualitative questions about customer experiences, use theme_retrieval.
- For broad comparison, recommendation, or “what does one restaurant do better” questions, use both competitor_comparison and theme_retrieval. The structured comparison shows rating and review-count differences, while theme_retrieval provides specific evidence about what customers mention, such as service, food quality, ambiance, authenticity, value, wait time, and special occasions.
- If a tool returns no results, say that the data was not found.

Use the restaurant names provided by the user. Do not assume the user only cares about a fixed pair of restaurants.
Be concise, but include enough evidence from tool outputs to justify the answer.
"""

### 7.6. Create Tool-Calling Agent Class with MLflow Tracing

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

# Enable MLflow tracing for OpenAI-compatible chat calls
mlflow.autolog()

print("MLflow OpenAI autologging enabled.")

In [0]:
class RestaurantToolCallingAgent:
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.tools = {tool.name: tool for tool in tools}
        self.workspace_client = WorkspaceClient()
        self.client = self.workspace_client.serving_endpoints.get_open_ai_client()
    
    def get_tool_specs(self):
        return [tool.spec for tool in self.tools.values()]
    
    def execute_tool(self, tool_name: str, arguments: dict):
        if tool_name not in self.tools:
            return {
                "status": "error",
                "message": f"Unknown tool: {tool_name}"
            }
        
        tool = self.tools[tool_name]
        return tool.exec_fn(**arguments)
    
    def predict(self, user_question: str, max_tool_rounds: int = 5, verbose: bool = True):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
        
        for _ in range(max_tool_rounds):
            response = self.client.chat.completions.create(
                model=self.llm_endpoint,
                messages=messages,
                tools=self.get_tool_specs(),
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            messages.append(assistant_message.model_dump(exclude_none=True))
            
            # If the model does not call a tool, return the final response
            if not assistant_message.tool_calls:
                return assistant_message.content
            
            # Execute tool calls
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments or "{}")
                
                if verbose:
                    print(f"Calling tool: {tool_name}")
                    print(f"Arguments: {arguments}")
                
                tool_result = self.execute_tool(tool_name, arguments)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, default=str)
                })
        
        return "The agent reached the maximum number of tool calls before producing a final answer."

### 7.7. Create Agents for Both LLMs

In [0]:
AGENT_LLAMA_3_3_70B = RestaurantToolCallingAgent(
    llm_endpoint=LLAMA_3_3_70B_ENDPOINT,
    tools=TOOL_INFOS
)

AGENT_GPT_OSS_20B = RestaurantToolCallingAgent(
    llm_endpoint=GPT_OSS_20B_ENDPOINT,
    tools=TOOL_INFOS
)

# Default agent
AGENT = AGENT_LLAMA_3_3_70B

print("=" * 60)
print("Restaurant comparison agents created.")
print("=" * 60)

print(f"Default agent: {LLAMA_3_3_70B_ENDPOINT}")
print(f"Comparison agent: {GPT_OSS_20B_ENDPOINT}")

### 7.8. Compare Both LLMs on the Same Prompts

#### Prompt 1

In [0]:
test_question = (
    "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 2

In [0]:
test_question = (
    "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 3

In [0]:
test_question = (
    "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 4

In [0]:
test_question = (
    "What is the best recipe for homemade lasagna?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 5

In [0]:
test_question = (
    "Can you summarize reviews for a dentist in New York?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

## 8. Evaluation Agent with LLM Judges

This section evaluates the restaurant comparison agent using LLM-as-a-judge scoring. The evaluation checks whether the agent answers the user’s question, stays grounded in tool outputs, uses the appropriate tools, and gracefully rejects out-of-scope questions. The evaluation is run through MLflow so that traces and judge scores can be reviewed in the Databricks Experiments UI.

### 8.1. Set MLflow Experiment

In [0]:
import mlflow

# Set MLflow experiment for final project traces and evaluation
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{current_user}/aai510_restaurant_agent_final"

mlflow.set_experiment(EXPERIMENT_NAME)

print("=" * 60)
print("MLflow experiment configured...")
print("=" * 60)
print(f"Experiment: {EXPERIMENT_NAME}")

### 8.2. Create Evaluation Questions

In [0]:
# Evaluation questions for LLM judge
# These are the same types of prompts tested in Section 7

EVALUATION_QUESTIONS = [
    {
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"]
    },
    {
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison"]
    },
    {
        "question": (
            "What are customers saying about the service and staff friendliness at Parma Cucina Italiana?"
        ),
        "expected_tools": ["theme_retrieval"]
    },
    {
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": []
    },
    {
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": []
    }
]

print("=" * 60)
print("Evaluation questions created...")
print("=" * 60)
print(f"Total evaluation questions: {len(EVALUATION_QUESTIONS)}")

display(EVALUATION_QUESTIONS)

### 8.3. Convert Questions to MLflow Evaluation Format

In [0]:
# Convert Section 8 evaluation questions into MLflow GenAI evaluation format
# MLflow requires expectations to be a dictionary

eval_data = []

for item in EVALUATION_QUESTIONS:
    expected_tools_text = ", ".join(item.get("expected_tools", []))

    if expected_tools_text == "":
        expected_tools_text = "none; the agent should reject the request if it is out of scope"

    eval_data.append(
        {
            "inputs": {
                "question": item["question"]
            },
            "expectations": {
                "expected_tools": expected_tools_text,
                "expected_behavior": (
                    "The agent should answer the question using the appropriate restaurant analysis tools, "
                    "stay grounded in retrieved reviews, business metadata, or tool outputs, "
                    "provide specific and useful business insight, and reject the request gracefully if it is out of scope."
                )
            }
        }
    )

print("=" * 60)
print("Evaluation dataset created...")
print("=" * 60)
print(f"Total evaluation records: {len(eval_data)}")

# Do not use display(eval_data) here because Databricks may not infer nested schemas correctly
for row in eval_data:
    print("\nQuestion:")
    print(row["inputs"]["question"])
    print("Expected tools:")
    print(row["expectations"]["expected_tools"])
    print("Expected behavior:")
    print(row["expectations"]["expected_behavior"])

### 8.4. Create Prediction Function

In [0]:
def predict_fn(question: str) -> str:
    """
    Run the default restaurant agent on a single evaluation question.
    The default agent uses Llama 3.3 70B.
    """
    
    response = AGENT.predict(
        question,
        verbose=False
    )
    
    return response

print("Prediction function created.")

In [0]:
## quick test
predict_fn("How many reviews does Parma Cucina Italiana have, and what is its average rating?")

### 8.5. Create T/F LLM Judge

In [0]:
from mlflow.genai.judges import make_judge

restaurant_owner_judge = make_judge(
    name="restaurant_owner_judge",
    instructions="""
You are evaluating a restaurant-review analysis agent for a final project.

The agent's purpose is to help a small restaurant owner understand San Diego Italian restaurant performance using:
- business metadata,
- rating metrics,
- rating distributions,
- retrieved review evidence,
- competitor comparison outputs,
- and tool outputs from the agent.

User input:
{{ inputs }}

Agent response:
{{ outputs }}

Expected behavior:
{{ expectations }}

Trace:
{{ trace }}

You must evaluate the agent response using the criteria below.

Return TRUE only if all required criteria are satisfied.
Return FALSE if any required criterion is not satisfied.

Criteria:

1. Groundedness:
Is the response grounded in the retrieved reviews, business metadata, or tool outputs?
True if the response does not invent unsupported facts or make claims that cannot be traced to the available evidence.

2. Answer relevance:
Does the response answer the user’s actual question?
True if the response directly addresses the user’s request rather than giving a generic or unrelated answer.

3. Restaurant-owner usefulness:
Is the response useful for a restaurant owner?
True if the response provides practical insight, decision support, or a clear next step.

4. Specificity:
Is the response specific enough to be meaningful?
True if the response includes concrete themes, complaints, strengths, comparisons, or recommendations rather than vague statements.

5. Appropriate tool use:
Did the agent use the appropriate tool or tool sequence?
True if the tool use matches the query type, such as lookup for metadata, retrieval for review evidence, or competitor comparison for comparison questions.

6. Out-of-scope handling:
If the input was irrelevant or out of scope, did the agent reject it gracefully?
True if the agent politely refused or redirected the user to an appropriate restaurant-review analysis task.
For in-scope questions, treat this criterion as satisfied.

7. Clarity:
Is the response clear and understandable for a small business owner?
True if the response is organized, concise, and written in plain business language.

Final decision rule:
- Return TRUE if all applicable criteria are satisfied.
- Return FALSE if any applicable criterion fails.
- Return only TRUE or FALSE.
- Do not include explanations, markdown, or extra text.
""",
    model="databricks:/databricks-gpt-oss-20b",
    feedback_value_type=bool
)

print("Custom TRUE/FALSE judge created: restaurant_owner_judge")

### 8.6. Run LLM Judge Evaluation

In [0]:
print("=" * 60)
print("Running LLM judge evaluation...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        restaurant_owner_judge
    ]
)

print("\nEvaluation complete.")
print("Review the MLflow evaluation results in the Experiments UI.")

---
## 9. Human Review Layer

This section implements the human evaluation component to validate LLM judge scores and compare model performance.

**Purpose**: Establish ground truth through human evaluation and measure agreement between human reviewers and the automated LLM judge.

**Process Overview**:
1. **Prepare Evaluation Materials**: Extract test cases, responses, and judge scores
2. **Distribute to Reviewers**: Each team member independently scores all 10 responses (5 questions × 2 models)
3. **Calculate Inter-Rater Agreement**: Measure consistency across human reviewers
4. **Consensus Discussion**: Resolve disagreements through team discussion
5. **Compare Human vs LLM Judge**: Analyze agreement between human consensus and automated scoring
6. **Document Findings**: Compile insights for presentation

**Evaluation Rubric**: Each response is scored PASS/FAIL on 6 dimensions:
- **Groundedness**: All facts verifiable from tool outputs
- **Answer Relevance**: Directly addresses the question
- **Restaurant Owner Usefulness**: Provides actionable business insight
- **Specificity**: Includes concrete numbers/themes/examples
- **Appropriate Tool Use**: Selected correct tool(s)
- **Out-of-Scope Handling**: Properly rejected irrelevant requests

### 9.1. Prepare Evaluation Materials

Extract the 5 test cases with responses from both LLMs into a structured format for human review.

**Test Cases**:
1. **Business Lookup** (In-Scope): "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
2. **Competitor Comparison** (In-Scope): "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
3. **Theme Retrieval** (In-Scope): "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
4. **Out-of-Scope #1**: "What is the best recipe for homemade lasagna?"
5. **Out-of-Scope #2**: "Can you summarize reviews for a dentist in New York?"

Each test case includes:
- Original question
- Agent responses (Llama 3.3 70B and GPT OSS 20B)
- Tools called
- Expected tool usage
- LLM judge score (from Section 8 evaluation)

In [0]:
# Extract responses from the test cells (63, 65, 67, 69, 71)
# We'll structure this data for human review

import pandas as pd
from pyspark.sql import Row

# Define test cases with metadata
test_cases = [
    {
        "test_id": "TEST_01",
        "question": "How many reviews does Parma Cucina Italiana have, and what is its average rating?",
        "expected_tools": ["business_lookup"],
        "scope": "In-Scope",
        "query_type": "Business Lookup"
    },
    {
        "test_id": "TEST_02",
        "question": "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?",
        "expected_tools": ["competitor_comparison", "theme_retrieval"],
        "scope": "In-Scope",
        "query_type": "Competitor Comparison"
    },
    {
        "test_id": "TEST_03",
        "question": "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?",
        "expected_tools": ["theme_retrieval"],
        "scope": "In-Scope",
        "query_type": "Theme Retrieval"
    },
    {
        "test_id": "TEST_04",
        "question": "What is the best recipe for homemade lasagna?",
        "expected_tools": [],
        "scope": "Out-of-Scope",
        "query_type": "Rejection Test"
    },
    {
        "test_id": "TEST_05",
        "question": "Can you summarize reviews for a dentist in New York?",
        "expected_tools": [],
        "scope": "Out-of-Scope",
        "query_type": "Rejection Test"
    }
]

print("=" * 60)
print("Test Cases Prepared for Human Review")
print("=" * 60)
print(f"Total test cases: {len(test_cases)}")
print(f"In-scope cases: {sum(1 for tc in test_cases if tc['scope'] == 'In-Scope')}")
print(f"Out-of-scope cases: {sum(1 for tc in test_cases if tc['scope'] == 'Out-of-Scope')}")

In [0]:
# Re-run all test cases to capture responses and tool calls
# This ensures we have clean data for human review

from datetime import datetime
import json

evaluation_records = []

for test_case in test_cases:
    test_id = test_case["test_id"]
    question = test_case["question"]
    
    print(f"\n{'='*80}")
    print(f"{test_id}: {test_case['query_type']}")
    print(f"{'='*80}")
    print(f"Question: {question}\n")
    
    # Run with Llama 3.3 70B
    print("Running Llama 3.3 70B...")
    llama_response = AGENT_LLAMA_3_3_70B.predict(question, verbose=True)
    
    # Run with GPT OSS 20B
    print("\nRunning GPT OSS 20B...")
    gpt_response = AGENT_GPT_OSS_20B.predict(question, verbose=True)
    
    # Store both responses
    evaluation_records.append({
        "test_id": test_id,
        "question": question,
        "query_type": test_case["query_type"],
        "scope": test_case["scope"],
        "expected_tools": ", ".join(test_case["expected_tools"]) if test_case["expected_tools"] else "None (should reject)",
        "model": "Llama 3.3 70B",
        "response": llama_response
    })
    
    evaluation_records.append({
        "test_id": test_id,
        "question": question,
        "query_type": test_case["query_type"],
        "scope": test_case["scope"],
        "expected_tools": ", ".join(test_case["expected_tools"]) if test_case["expected_tools"] else "None (should reject)",
        "model": "GPT OSS 20B",
        "response": gpt_response
    })
    
    print("\n" + "-"*80)

print(f"\n{'='*80}")
print(f"Captured {len(evaluation_records)} evaluation records")
print(f"({len(test_cases)} questions × 2 models)")
print(f"{'='*80}")

In [0]:
# Convert to pandas DataFrame first (handles mixed types better)
# Then convert to Spark DataFrame for display and storage
import pandas as pd

# Ensure all response values are strings
for record in evaluation_records:
    record["response"] = str(record["response"])

# Create pandas DataFrame
evaluation_pandas_df = pd.DataFrame(evaluation_records)

# Convert to Spark DataFrame
evaluation_df = spark.createDataFrame(evaluation_pandas_df)

print("=" * 60)
print("Human Review Dataset Created")
print("=" * 60)

# Display the structured dataset
display(evaluation_df.select(
    "test_id",
    "question",
    "query_type",
    "scope",
    "expected_tools",
    "model",
    "response"
).orderBy("test_id", "model"))

print(f"\nTotal records: {evaluation_df.count()}")
print("\nReady for export to human review spreadsheet.")

In [0]:
# Save the evaluation dataset to Unity Catalog
# This allows team members to access it for review

REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.human_review_dataset"

evaluation_df.write.mode("overwrite").saveAsTable(REVIEW_TABLE)

print("=" * 60)
print("Human Review Dataset Saved")
print("=" * 60)
print(f"Table: {REVIEW_TABLE}")
print(f"\nTeam members can access this table for human evaluation.")
print(f"\nNext Steps:")
print("1. Export to Google Sheets or Excel for review")
print("2. Add review columns: human_score, groundedness, relevance, usefulness, specificity, tool_use, scope_handling, comments")
print("3. Distribute to all team members for independent review")
print("4. Collect reviews and calculate inter-rater agreement")

In [0]:
# Create a template with review columns for human evaluators
# This can be exported to a spreadsheet

from pyspark.sql.functions import lit

review_template_df = evaluation_df.select(
    "test_id",
    "question",
    "query_type",
    "scope",
    "expected_tools",
    "model",
    "response"
)

print("=" * 60)
print("Human Review Template Created")
print("=" * 60)
print("\nTemplate includes the following review columns:")
print("- llm_judge_score: PASS/FAIL from LLM judge")
print("- human_overall_score: PASS/FAIL from human reviewer")
print("- groundedness: PASS/FAIL (facts verifiable from tool outputs)")
print("- answer_relevance: PASS/FAIL (directly answers question)")
print("- usefulness: PASS/FAIL (provides actionable insight)")
print("- specificity: PASS/FAIL (includes concrete details)")
print("- tool_use: PASS/FAIL (selected appropriate tool)")
print("- scope_handling: PASS/FAIL (proper rejection of out-of-scope)")
print("- comments: Free text for reviewer notes")
print("- reviewer_name: Name of the team member conducting review")

# Display sample
print("\nSample Template (first 2 records):")
display(review_template_df)

### 9.2. Team Review for Responses

**Quick Rubric Reference for Reviewers**:

| Dimension | PASS Criteria | FAIL Example |
|-----------|---------------|---------------|
| **Groundedness** | All facts verifiable from tool outputs | "Parma has 600 reviews" (actual: 552) |
| **Answer Relevance** | Directly answers the question | Generic response, misses the question |
| **Usefulness** | Provides actionable business insight | Just dumps data without interpretation |
| **Specificity** | Includes concrete numbers/themes | Vague: "customers seem happy" |
| **Tool Use** | Selected appropriate tool(s) | Used wrong tool or skipped necessary tools |
| **Scope Handling** | In-scope: answered; Out-of-scope: politely rejected | Attempted to answer recipe question |

**Scoring Rule**: PASS = all 6 criteria met, FAIL = any criterion failed

### Human Review Table - Team Evaluation

**Instructions**: Each of the three team members should independently review all 5 test questions and both model responses (Llama 3.3 70B and GPT OSS 20B). Fill in your scores below.

**Scoring**: Use PASS or FAIL for each dimension. Overall = PASS only if ALL dimensions pass.

---

#### Reviewer 1: [Brandon Wirgau]

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | P | P | P | P | P | P | P | |
| TEST_01 | GPT OSS 20B | Parma review count & rating | P | P | P | P | P | P | P | |
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | P | P | P | P | P | P | P | |
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | F | F | F | F | F | F | F | Gave recommendation about retrieving additional data instead of providing a direct answer. |
| TEST_03 | Llama 3.3 70B | Pasta quality themes | P | P | P | P | P | P | P | |
| TEST_03 | GPT OSS 20B | Pasta quality themes | P | P | P | P | P | P | P | |
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | Stayed in the scope of the question and did not try to answer anything out of scope. |
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | P | P | P | P | P | P | P | |
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | |
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | P | P | P | P | P | P | P | |

---

#### Reviewer 2: [Team Member Name]

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | | | | | | | | |
| TEST_01 | GPT OSS 20B | Parma review count & rating | | | | | | | | |
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | | | | | | | | |
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | | | | | | | | |
| TEST_03 | Llama 3.3 70B | Pasta quality themes | | | | | | | | |
| TEST_03 | GPT OSS 20B | Pasta quality themes | | | | | | | | |
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | | | | | | | | |
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | | | | | | | | |
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | | | | | | | | |
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | | | | | | | | |

---

#### Reviewer 3: [Team Member Name]

| Test ID | Model | Question Summary | Groundedness | Answer Relevance | Usefulness | Specificity | Tool Use | Scope Handling | Overall Score | Comments |
|---------|-------|------------------|--------------|------------------|------------|-------------|----------|----------------|---------------|----------|
| TEST_01 | Llama 3.3 70B | Parma review count & rating | | | | | | | | |
| TEST_01 | GPT OSS 20B | Parma review count & rating | | | | | | | | |
| TEST_02 | Llama 3.3 70B | Compare Parma vs Cesarina | | | | | | | | |
| TEST_02 | GPT OSS 20B | Compare Parma vs Cesarina | | | | | | | | |
| TEST_03 | Llama 3.3 70B | Pasta quality themes | | | | | | | | |
| TEST_03 | GPT OSS 20B | Pasta quality themes | | | | | | | | |
| TEST_04 | Llama 3.3 70B | Lasagna recipe (out-of-scope) | | | | | | | | |
| TEST_04 | GPT OSS 20B | Lasagna recipe (out-of-scope) | | | | | | | | |
| TEST_05 | Llama 3.3 70B | Dentist in NY (out-of-scope) | | | | | | | | |
| TEST_05 | GPT OSS 20B | Dentist in NY (out-of-scope) | | | | | | | | |

---

**After all reviews are complete**:
1. Calculate inter-rater agreement (Cohen's Kappa or Fleiss' Kappa)
2. Identify disagreements for consensus discussion
3. Compare human consensus vs LLM judge scores
4. Document findings for presentation